[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adkipnis/metabeta/blob/main/demos/03-priors.ipynb)

# priors and model comparison

metabeta takes the prior specification as an input, and permits model comparison via Bayes
factors.<br>
For demonstration purposes, this notebook fits simulated data under known ground truth parameters.

In [ ]:
%pip install -q metabeta==0.6.0

## Load a checkpoint

In [1]:
import numpy as np
import pandas as pd
import torch

from metabeta import Api

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mb = Api.from_pretrained('gaussian', device=device)  # the test score is continuous

Resolving metabeta 'normal' checkpoint from Hugging Face (adkipnis/metabeta@v1)...


Using checkpoint: /Users/alex/.cache/huggingface/hub/models--adkipnis--metabeta/snapshots/66c398ef31d65400f9ce6cff262265a9cd895a4d/metabeta-normal.pt


Loading metabeta checkpoint on cpu...


Metabeta checkpoint ready.


## Simulate the data

Imagine that students in 15 schools take a test:

- **Outcome**: the test `score`.
- **Predictors**: weekly study `hours` (which raise the score) and the `commute` in minutes
  (which doesn't).
- **Groups**: each school gets its own intercept and slope.

In [2]:
rng = np.random.default_rng(1)
n_schools = 15
school = np.repeat(np.arange(n_schools), rng.integers(6, 13, size=n_schools))  # (n,) 6–12 each
hours = rng.normal(5, 2, size=school.size).clip(0)
commute = rng.normal(30, 10, size=school.size).clip(5)

truth = pd.Series(
    {
        'Intercept': 60.0,
        'hours': 2.0,
        'commute': 0.0,
        'σ_Intercept': 6.0,
        'σ_hours': 1.0,
        'Corr(Intercept, hours)': -0.3,
        'σ_Residual': 8.0,
    }
)
sd_school = truth[['σ_Intercept', 'σ_hours']].to_numpy()
corr = np.array([[1, truth['Corr(Intercept, hours)']], [truth['Corr(Intercept, hours)'], 1]])
u = rng.multivariate_normal(np.zeros(2), corr * np.outer(sd_school, sd_school), size=n_schools)
score = (
    truth['Intercept']
    + u[school, 0]
    + (truth['hours'] + u[school, 1]) * hours
    + truth['commute'] * commute
    + rng.normal(0, truth['σ_Residual'], size=school.size)
)
df = pd.DataFrame(
    {'score': score, 'hours': hours, 'commute': commute, 'school': [f's{g:02d}' for g in school]}
)

sizes = df.groupby('school').size()
print(f'{len(df)} students in {len(sizes)} schools with {sizes.min()}–{sizes.max()} students each')

137 students in 15 schools with 6–12 students each


## Specifying priors

`sample(priors=...)` takes a dictionary with up to four keys.<br>
Terms are named as in the formula. Anything unspecified keeps its default.

| key | parameters | per term | supported families | default |
|---|---|---|---|---|
| `fixed` | intercept and slopes | `{'nu': mean, 'tau': scale}` | `'normal'`, `'student'` | N(0, 2.5) |
| `random_sd` | standard deviation (SD) of each random effect | `{'tau': scale}` | `'halfnormal'`, `'halfstudent'`, `'exponential'` | HN(2.5) |
| `sigma_eps` | residual SD | `{'tau': scale}`, without terms | as `random_sd` | HT₅(1) |
| `corr_rfx` | correlations of the random effects | `{'eta': concentration}`, without terms | LKJ | LKJ(1) |

- A `'family'` entry sets the family of all the key's terms.
- N and HN are the normal and half-normal.
- t₅ and HT₅ are the Student-t and half-Student-t with five degrees of freedom.
- An LKJ (Lewandowski–Kurowicka–Joe) concentration above 1 favors weak correlations.
- `sigma_eps` exists for the Gaussian family only, `corr_rfx` from two random effects onwards.
- The defaults are [bambi](https://bambinos.github.io/bambi/)'s weakly informative priors.

`sample` standardizes each numeric predictor to mean 0 and SD 1, as recommended for weakly
informative priors ([Gelman et al., 2008](https://doi.org/10.1214/08-AOAS191)).<br>
The priors live on that scale, since some of them have no closed form in the data's original
units.<br>
The summary reads them back in the data's units.

Let's add two priors besides the default:

- **informed**: draws on a pilot study that reported a standardized `hours` effect of about 0.3.
- **skeptical**: doubts that study time matters.

In [3]:
informed = {
    'fixed': {'family': 'student', 'hours': {'nu': 0.3, 'tau': 0.2}},  # around the pilot study
    'random_sd': {'family': 'exponential', 'Intercept': {'tau': 0.5}, 'hours': {'tau': 0.2}},
    'sigma_eps': {'family': 'halfnormal', 'tau': 1.0},
    'corr_rfx': {'eta': 2},
}
skeptical = {'fixed': {'hours': {'tau': 0.2}}}  # narrow, around 0
priors = {'default': None, 'informed': informed, 'skeptical': skeptical}  # None: the defaults

## Multiple priors in one call

A dictionary of named priors fits all in one call (one row per prior).

In [4]:
%%time
formula = 'score ~ hours + commute + (1 + hours | school)'
result = mb.sample(df, formula=formula, priors=priors, seed=0)  # seed for reproducibility

CPU times: user 208 ms, sys: 15.2 ms, total: 224 ms
Wall time: 157 ms


### Posterior summary

`posteriorSummary(result, prior_id=...)` prints one prior's row, by name or by index.

In [5]:
print(mb.posteriorSummary(result, prior_id='informed'))

Formula:  score ~ hours + commute + (1 + hours | school)
n = 137   m = 15
Priors:
  Intercept ~ t₅(71.2, 23.57) at mean hours, commute
  hours ~ t₅(1.591, 1.061)
  commute ~ t₅(0, 2.432)
  σ_Intercept ~ Exp(4.713) at mean hours
  σ_hours ~ Exp(1.061)
  Corr ~ LKJ(2) at mean hours
  σ_Residual ~ HN(9.426)

Fixed Effects:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |   P(>0) |   Contr. |
|-----------|--------|-------|--------|---------|---------|---------|----------|
| Intercept | 63.164 | 3.178 | 56.063 |  63.316 |  69.136 |   1.000 |    0.999 |
| hours     |  2.023 | 0.373 |  1.329 |   2.016 |   2.725 |   1.000 |    0.926 |
| commute   | -0.057 | 0.067 | -0.185 |  -0.058 |   0.076 |   0.196 |    1.000 |

Standard Deviations:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |
|-----------|--------|-------|--------|---------|---------|
| Intercept |  5.959 | 1.631 |  3.208 |   5.771 |   9.837 |
| hours     |  0.387 | 0.325 |  0.013 |   0.315 |   1.228 |
| Residual 

### Results side by side

The table lines up each prior's posterior means and 95% intervals with the true values:

- Every interval contains the true value.
- The skeptical prior is narrow enough to pull `hours` toward 0.
- `σ_hours` moves most: 15 schools say little about how the slope varies.

In [6]:
def estimates(name):
    """Posterior mean [95% interval] of a prior's global parameters, in the data's units."""
    draws = mb.posteriorDraws(result, prior_id=name)  # one column per parameter
    mean, lo, hi = draws.mean(), draws.quantile(0.025), draws.quantile(0.975)
    return pd.Series([f'{m:.2f} [{l:.2f}, {h:.2f}]' for m, l, h in zip(mean, lo, hi)], mean.index)


side_by_side = pd.DataFrame({name: estimates(name) for name in priors})
side_by_side.insert(0, 'truth', truth)
side_by_side.rename_axis('mean [95% interval]')

,truth,default,informed,skeptical
mean [95% interval],,,,
Intercept,60.0,"62.92 [57.28, 68.89]","63.16 [56.06, 69.14]","63.77 [57.86, 69.50]"
hours,2.0,"2.11 [1.34, 2.90]","2.02 [1.33, 2.73]","1.85 [1.11, 2.64]"
commute,0.0,"-0.06 [-0.20, 0.06]","-0.06 [-0.19, 0.08]","-0.06 [-0.18, 0.08]"
σ_Intercept,6.0,"6.44 [2.84, 12.58]","5.96 [3.21, 9.84]","6.42 [2.44, 11.00]"
σ_hours,1.0,"0.57 [0.02, 1.61]","0.39 [0.01, 1.23]","0.56 [0.02, 1.66]"
σ_Residual,8.0,"7.25 [6.40, 8.24]","7.35 [6.46, 8.52]","7.35 [6.48, 8.44]"
"Corr(Intercept, hours)",-0.3,"-0.15 [-0.91, 0.90]","-0.18 [-0.85, 0.73]","-0.21 [-0.96, 0.90]"


---

## Model comparison

So far, the model stayed the same and only its priors changed.<br>
Now we ask whether the model needs `commute` at all.<br>
A Bayes factor (BF) answers this: it compares how probable the data are under two models.

`evidence=True` adds `result.evidence`:

- **`log_evidence`**: the log marginal likelihood $\log p(y)$, one per row.<br>
  It measures how probable the data are under the row's model.
- **`psis_k`**: the estimate's Pareto $\hat k$, from Pareto-smoothed importance sampling (PSIS).<br>
  `sample` warns when `psis_k` $>0.7$, indicating a poor proposal.

Note: A BF depends on the prior of any parameter that only one of the models has.<br>
We fit the model with `commute` under three priors for its slope:

In [7]:
# no commute
result_without = mb.sample(
    df, formula='score ~ hours + (1 + hours | school)', evidence=True, seed=0
)

# commute under varying priors
commute_priors = {  # the slope's prior SD
    'default': None,
    'medium': {'fixed': {'commute': {'tau': 0.5}}},
    'narrow': {'fixed': {'commute': {'tau': 0.1}}},
}
result_with = mb.sample(df, formula=formula, priors=commute_priors, evidence=True, seed=0)

`bayesFactor` returns the BF of the first model over the second, per prior.<br>
The log BF is the difference of the two models' `log_evidence`. Above 0, it favors leaving
`commute` out.<br>
Its strength follows [Kass and Raftery (1995)](https://doi.org/10.1080/01621459.1995.10476572).

In [8]:
mb.bayesFactor(result_without, result_with).round(2)

,log BF,BF,strength,PSIS k̂
prior,,,,
default,3.15,23.37,strong,0.24
medium,1.57,4.79,positive,0.46
narrow,0.25,1.28,weak,0.40


Every BF favors leaving `commute` out; by how much depends on the slope's prior:

- The default prior spreads the slope over values the data rule out.
- A narrower prior brings the two models' predictions closer, so the BF shrinks.

We recommend reporting a Bayes factor with the prior it came from.